# 4. Load – silver ➜ golden
Construye las tablas de negocio a partir de `silver.car_sales_enriched`:

| Tabla golden | Pregunta de negocio | Técnica destacada |
|---|---|---|
| `ventas_mensuales_estado` | ¿Cómo evolucionan las ventas por estado mes a mes? | Ventana con `lag` |
| `ranking_modelos` | ¿Cuáles son los modelos más vendidos cada año? | Ventana con `dense_rank` |
| `kpi_concesionarios` | ¿Qué concesionario vende más y cuál es su modelo estrella? | `row_number` + join + participación % |
| `segmentos_clientes` | ¿Qué compra cada segmento de clientes? | Agregación multidimensional |

In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalogo", "vehiculos")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_ventas = spark.table(f"{catalogo}.{esquema_source}.car_sales_enriched").cache()
print(f"Ventas en silver: {df_ventas.count()}")

In [0]:
def cargar_golden(df, tabla):
    """Escribe el DataFrame en la tabla golden respetando el orden de columnas de la tabla."""
    destino = f"{catalogo}.{esquema_sink}.{tabla}"
    columnas = spark.table(destino).columns
    df.select(*columnas).write.mode("overwrite").insertInto(destino)
    print(f"{destino}: {spark.table(destino).count()} filas")

## 1. Ventas mensuales por estado
`lag` obtiene el monto del mes anterior dentro de cada estado para calcular la variación mes a mes (MoM).

In [0]:
df_mensual = df_ventas.groupBy("sale_year", "sale_month", "state_name").agg(
    count("car_id").alias("unidades"),
    F.sum("price").cast(LongType()).alias("monto_total"),
    F.round(F.avg("price"), 2).alias("ticket_promedio")
)

ventana_estado = Window.partitionBy("state_name").orderBy("sale_year", "sale_month")

df_mensual = df_mensual \
    .withColumn("monto_mes_anterior", lag("monto_total").over(ventana_estado)) \
    .withColumn("variacion_mom_pct",
                F.round((col("monto_total") - col("monto_mes_anterior")) / col("monto_mes_anterior") * 100, 2)) \
    .orderBy("state_name", "sale_year", "sale_month")

display(df_mensual)

In [0]:
cargar_golden(df_mensual, "ventas_mensuales_estado")

## 2. Ranking anual de modelos
`dense_rank` por año ordenado por monto vendido.

In [0]:
df_modelos = df_ventas.groupBy("sale_year", "company", "model").agg(
    count("car_id").alias("unidades"),
    F.sum("price").cast(LongType()).alias("monto_total"),
    F.round(F.avg("price"), 2).alias("precio_promedio")
)

ventana_anio = Window.partitionBy("sale_year").orderBy(col("monto_total").desc())

df_ranking = df_modelos.withColumn("ranking_anual", dense_rank().over(ventana_anio))

display(df_ranking.filter(col("ranking_anual") <= 10).orderBy("sale_year", "ranking_anual"))

In [0]:
cargar_golden(df_ranking, "ranking_modelos")

## 3. KPI por concesionario
- **Participación %**: monto del concesionario sobre el total vendido.
- **Modelo top**: modelo con más unidades vendidas por concesionario (`row_number`).

In [0]:
monto_global = df_ventas.agg(F.sum("price")).collect()[0][0]

df_kpi = df_ventas.groupBy("dealer_name").agg(
    count("car_id").alias("unidades"),
    F.sum("price").cast(LongType()).alias("monto_total"),
    F.round(F.avg("price"), 2).alias("ticket_promedio")
).withColumn("participacion_pct", F.round(col("monto_total") / lit(monto_global) * 100, 2))

In [0]:
ventana_dealer = Window.partitionBy("dealer_name").orderBy(col("unidades_modelo").desc(), col("modelo"))

df_modelo_top = df_ventas \
    .withColumn("modelo", concat_ws(" ", col("company"), col("model"))) \
    .groupBy("dealer_name", "modelo").agg(count("car_id").alias("unidades_modelo")) \
    .withColumn("rn", row_number().over(ventana_dealer)) \
    .filter(col("rn") == 1) \
    .select("dealer_name", col("modelo").alias("modelo_top"))

df_kpi = df_kpi.join(df_modelo_top, "dealer_name", "left").orderBy(col("monto_total").desc())

display(df_kpi)

In [0]:
cargar_golden(df_kpi, "kpi_concesionarios")

## 4. Segmentos de clientes

In [0]:
df_segmentos = df_ventas.groupBy("income_segment", "gender", "body_style").agg(
    count("car_id").alias("unidades"),
    F.sum("price").cast(LongType()).alias("monto_total"),
    F.round(F.avg("price"), 2).alias("ticket_promedio")
).orderBy("income_segment", "gender", col("unidades").desc())

display(df_segmentos)

In [0]:
cargar_golden(df_segmentos, "segmentos_clientes")

## 5. Optimización de las tablas golden
Compactación de archivos pequeños con la API de Delta (`OPTIMIZE`), para acelerar las consultas del dashboard.

In [0]:
for tabla in ["ventas_mensuales_estado", "ranking_modelos", "kpi_concesionarios", "segmentos_clientes"]:
    DeltaTable.forName(spark, f"{catalogo}.{esquema_sink}.{tabla}").optimize().executeCompaction()
    print(f"OPTIMIZE ok: {tabla}")

df_ventas.unpersist()